# BlackBox — Phase 3: The Read Path

Phase 2 called `search()` and used whatever came back. This notebook opens up *how* mem0
decides what counts as relevant -- the actual read path from the original design doc:

**query embedding → similarity search → ranking → injection**

We can't see raw embedding vectors (MemoryClient is hosted, that math happens server-side),
but we *can* see and control the real levers mem0 exposes: `top_k`, `threshold`, `rerank`,
and the per-result `score`. This notebook works with those directly instead of guessing at
internals we don't have access to.

Same `eng_01` data from Phases 1 and 2 -- nothing gets reset.


## 0. Reconnect, and rule out a confound

mem0 has a project-level **Memory Decay** setting that reranks search results by recency.
If it's on, it'll mix into the scores we're about to look at, and we won't be able to tell
whether a score changed because of relevance or because of decay. We turn it off here so this
notebook is a clean test of ranking-by-relevance alone. (We'll turn decay back on deliberately
in the next notebook, where it's the actual thing we're testing.)


In [1]:
import os
from dotenv import load_dotenv
from mem0 import MemoryClient

load_dotenv()
client = MemoryClient(api_key=os.getenv("MEM0_API_KEY"))

# Make sure decay isn't quietly reranking our results for this notebook
client.project.update(decay=False)

existing = client.get_all(filters={"user_id": "eng_01"})
print(f"{len(existing.get('results', []))} memories on file for eng_01")

7 memories on file for eng_01


## 1. Scores, not just text

Every result from `search()` carries a relevance `score` alongside the memory text.
"Relevant" isn't binary -- it's a ranked number, and this is the first time we've looked at it.


In [2]:
query = "what hydraulic system do I work on?"

results = client.search(query=query, filters={"user_id": "eng_01"})

for r in results.get("results", []):
    print(f"{r['score']:.3f}  {r['memory']}")

0.315  User works on Boeing 737 hydraulic systems, handling their maintenance and engineering aspects
0.291  User has transitioned from working on Boeing 737 hydraulic systems to now working on the Boeing 787 fleet
0.279  The Boeing 737 hydraulic system includes System A and System B as primary systems and a Standby system as the backup hydraulic system
0.260  The backup for the Boeing 737 hydraulic system is the Standby Hydraulic System, a separate electrically‑powered pump usually driven by the APU or an electric motor, which can take over if System A or System B fails and is isolated from the primary systems, used only when a primary system is out of service or during a loss‑of‑pressure event.
0.237  User is seeking guidance on how often to check hydraulic fluid levels for their fleet of Boeing 737 aircraft
0.223  User observed a low hydraulic pressure gauge reading during preflight checks on a Boeing 737 and asked for troubleshooting steps
0.207  User observed a low hydraulic press

## 2. Threshold experiment

`threshold` is the minimum score a memory needs to be included at all. Run the same query at
increasing thresholds and watch the result set shrink.


In [3]:
for threshold in [0.0, 0.3, 0.6, 0.9]:
    results = client.search(query=query, filters={"user_id": "eng_01"}, threshold=threshold)
    count = len(results.get("results", []))
    print(f"threshold={threshold} -> {count} results")

threshold=0.0 -> 7 results
threshold=0.3 -> 7 results
threshold=0.6 -> 7 results
threshold=0.9 -> 7 results


Pick one of the middling thresholds above and print what actually survived at that bar --
this is the concrete version of "minimum semantic relevance score" from the docs.


In [9]:
# 1. Perform search
results = client.search(query=query, filters={"user_id": "eng_01"})

# 2. Apply a realistic threshold filter in Python (e.g. 0.20)
THRESHOLD = 0.27

filtered_memories = [
    r for r in results.get("results", []) if r.get("score", 0) >= THRESHOLD
]

# 3. Print filtered memories
print(f"Found {len(filtered_memories)} matching memories (score >= {THRESHOLD}):")
for r in filtered_memories:
  print(f"{r['score']:.3f}  {r['memory']}")


Found 3 matching memories (score >= 0.27):
0.315  User works on Boeing 737 hydraulic systems, handling their maintenance and engineering aspects
0.291  User has transitioned from working on Boeing 737 hydraulic systems to now working on the Boeing 787 fleet
0.279  The Boeing 737 hydraulic system includes System A and System B as primary systems and a Standby system as the backup hydraulic system


## 3. top_k experiment

Same query, different caps on how many results come back. A low `top_k` can silently drop a
fact that actually mattered -- this is the exact design question from Phase 2's roadmap doc:
*how many memories should get injected into the prompt?*


In [6]:
for k in [1, 3, 10]:
    results = client.search(query=query, filters={"user_id": "eng_01"}, top_k=k)
    print(f"top_k={k}:")
    for r in results.get("results", []):
        print(f"   {r['score']:.3f}  {r['memory']}")
    print()

top_k=1:
   0.315  User works on Boeing 737 hydraulic systems, handling their maintenance and engineering aspects

top_k=3:
   0.315  User works on Boeing 737 hydraulic systems, handling their maintenance and engineering aspects
   0.291  User has transitioned from working on Boeing 737 hydraulic systems to now working on the Boeing 787 fleet
   0.279  The Boeing 737 hydraulic system includes System A and System B as primary systems and a Standby system as the backup hydraulic system

top_k=10:
   0.315  User works on Boeing 737 hydraulic systems, handling their maintenance and engineering aspects
   0.291  User has transitioned from working on Boeing 737 hydraulic systems to now working on the Boeing 787 fleet
   0.279  The Boeing 737 hydraulic system includes System A and System B as primary systems and a Standby system as the backup hydraulic system
   0.260  The backup for the Boeing 737 hydraulic system is the Standby Hydraulic System, a separate electrically‑powered pump usually 

## 4. Ranking, using the contradiction we already have

Phase 2 found that storage never deleted the old "works on 737" fact after the user said
they'd moved to 787. This is the cleanest real test of whether **ranking** is where mem0
resolves that conflict, even though storage didn't.


In [10]:
results = client.search(
    query="what hydraulic system do I work on?",
    filters={"user_id": "eng_01"},
    top_k=10,
)

for r in results.get("results", []):
    marker = "  <-- 787 (new)" if "787" in r["memory"] else ("  <-- 737 (stale)" if "737" in r["memory"] and "works on" in r["memory"].lower() else "")
    print(f"{r['score']:.3f}  {r['memory']}{marker}")

0.315  User works on Boeing 737 hydraulic systems, handling their maintenance and engineering aspects  <-- 737 (stale)
0.291  User has transitioned from working on Boeing 737 hydraulic systems to now working on the Boeing 787 fleet  <-- 787 (new)
0.279  The Boeing 737 hydraulic system includes System A and System B as primary systems and a Standby system as the backup hydraulic system
0.260  The backup for the Boeing 737 hydraulic system is the Standby Hydraulic System, a separate electrically‑powered pump usually driven by the APU or an electric motor, which can take over if System A or System B fails and is isolated from the primary systems, used only when a primary system is out of service or during a loss‑of‑pressure event.
0.237  User is seeking guidance on how often to check hydraulic fluid levels for their fleet of Boeing 737 aircraft
0.223  User observed a low hydraulic pressure gauge reading during preflight checks on a Boeing 737 and asked for troubleshooting steps
0.207  Use

Look at where the two facts land relative to each other. If the 787 fact consistently
outranks the stale 737 fact, that's evidence conflict resolution is happening at **read time**
via scoring, not at **write time** via deletion. If they're close together or the 737 fact
ranks higher, that's worth flagging as a real limitation, not glossing over.


## 5. Bonus: rerank

mem0 also supports a `rerank` flag that applies a secondary reranking pass. Quick before/after
on the same query.


In [ ]:
plain = client.search(query=query, filters={"user_id": "eng_01"}, top_k=5)
reranked = client.search(query=query, filters={"user_id": "eng_01"}, top_k=5, rerank=True)

print("rerank=False:")
for r in plain.get("results", []):
    print(f"   {r['score']:.3f}  {r['memory']}")

print("\nrerank=True:")
for r in reranked.get("results", []):
    print(f"   {r['score']:.3f}  {r['memory']}")

## 6. Injection comparison: does what you retrieve change what gets said?

Same question, three different retrieval settings, three generated answers. This is the
actual payoff of the read path -- showing that *what* you choose to inject materially changes
the assistant's answer, not just whether memory "works" in the abstract.


In [11]:
from lab_llm_config import complete

def answer_with_settings(question, **search_kwargs):
    results = client.search(query=question, filters={"user_id": "eng_01"}, **search_kwargs)
    memories = [r["memory"] for r in results.get("results", [])]
    memory_block = "\n".join(f"- {m}" for m in memories)
    prompt = (
        "You are an aircraft maintenance assistant. Use these facts if relevant, "
        "and answer naturally without mentioning 'stored memories':\n\n"
        f"{memory_block}\n\nQuestion: {question}"
    )
    return complete(prompt), memories

<frozen abc>:106: DeprecationWarning: BaseAgentConfig is deprecated and will be removed in future versions. Config is now loaded via reflection so the separate config class is no longer needed.


In [12]:
question = "what hydraulic system do I work on?"

settings = {
    "top_1": {"top_k": 1},
    "top_5": {"top_k": 5},
    "threshold_0.5": {"threshold": 0.5, "top_k": 10},
}

for label, kwargs in settings.items():
    answer, memories = answer_with_settings(question, **kwargs)
    print(f"--- {label} ({len(memories)} memories used) ---")
    print(answer)
    print()

--- top_1 (1 memories used) ---


--- top_5 (5 memories used) ---
You’re working on the **Boeing 737 hydraulic system**.  
That system is made up of two primary pumps—**System A** and **System B**—and a separate **Standby Hydraulic System** that serves as the backup. The standby pump is electrically powered (usually by the APU or an electric motor) and is isolated from the primary systems, coming into play only when one of the primary systems fails or during a loss‑of‑pressure event.

--- threshold_0.5 (7 memories used) ---
You’re now working on the hydraulic system for the Boeing 787 fleet.



## Wrap-up

What this notebook actually showed, not just claimed:

1. **Relevance is a number, not a flag** -- `score` gives you the real signal, `threshold`
   and `top_k` are the two levers that turn that signal into an actual result set.
2. **Ranking, not just storage, is where conflicts might get resolved.** Section 4 is the
   direct answer to the question Phase 2 left open.
3. **Retrieval settings change what the assistant says**, not just how many facts print to
   the console -- Section 6 is the version of this that's actually visible to an end user.

**Next up:** Notebook B, decay and importance -- now that we know mem0 ships a real, documented
decay feature (`client.project.update(decay=True)`), that notebook can test it directly instead
of guessing whether recency affects ranking at all.
